## Inference on shared models

Run inference on the shared sigmoid-attention models (and its softmax baseline) using AXLearn.

### Setup

Please see [Getting Started](https://github.com/apple/axlearn/blob/main/README.md) for the general getting started guide for AXLearn. Including how to define new trainer configs yourself, how to use the CLI, how to launch trainings and why certain design decisions were made.

To install required dependencies for this notebook, run:
```shell
pip install --ignore-installed "axlearn[core,apple-silicon,gcp] @ git+https://github.com/apple/axlearn.git"

pip install tabulate
```

And then, make sure to authenticate to GCP: `gcloud auth login && gcloud auth application-default login`.

In [1]:
# General imports.
from typing import List, Iterator, Tuple, Literal, Sequence

# JAX must be imported before tensorflow.
import jax
import jax.numpy as jnp
import seqio
from tqdm import tqdm
import os
import subprocess
from pathlib import Path

# Necessary so that the checkpoint loader works inside a notebook.
import nest_asyncio

nest_asyncio.apply()

In [8]:
# The LM configurations and checkpoints.
from axlearn.experiments import get_named_trainer_config
from axlearn.common.config import config_for_function

# Stuff to configure the local device mesh.
from axlearn.common import utils_spmd

# To control what to load from the saved checkpoint.
from axlearn.common import state_builder
from axlearn.common.checkpointer import CheckpointValidationType

# For the tokenizer/vocab.
from axlearn.experiments.text import common

# For typing stuff.
from axlearn.common.utils import DataPartitionType, set_data_dir
from axlearn.common import utils
from axlearn.common.inference import InferenceRunner


JAX_BACKEND: Literal["cpu", "tpu", "gpu"] = "cpu"
DATA_DIR: str = "gs://axlearn-public/tensorflow_datasets"

REMOTE_MODEL_DIR = "gs://axlearn-public/experiments/"
LOCAL_MODEL_DIR = "downloads/models/"
MODEL_INFO: dict[str, dict[str, str]] = {
    # Sigmoid-based attention.
    "7b-sigmoid": {
        "checkpoint_dir": "gala-7B-sigmoid-hybridnorm-alibi-sprp-2024-12-03-1002/checkpoints/step_00250000",
        "config_name": "gala-sigmoid-7B-4k-hybridnorm-alibi-sp-rp",
        "sentencepiece_model_name": "bpe_32k_c4.model",
        "config_module": "axlearn.experiments.text.gpt.pajama_sigmoid_trainer",
    },
    # Softmax baseline.
    "7b-softmax": {
        "checkpoint_dir": "gala-7B-hybridnorm-alibi-sprp-2024-12-02-1445/checkpoints/step_00250000",
        "config_name": "gala-7B-hybridnorm-alibi-flash-sp-rp",
        "sentencepiece_model_name": "bpe_32k_c4.model",
        "config_module": "axlearn.experiments.text.gpt.pajama_trainer",
    },
}

utils_spmd.setup(jax_backend=JAX_BACKEND)


def _init_state_builder_discard_optimizer(
    *,
    source_config_name: str,
    source_config_module: str,
    mesh_axis_names: Sequence[str],
    mesh_shape: Sequence[int],
    checkpoint_dir: str,
) -> state_builder.Builder.Config:
    converter = state_builder.ModelStateScopeConverter.default_config().set(
        source_trainer_config=config_for_function(get_named_trainer_config).set(
            config_name=source_config_name,
            config_module=source_config_module,
        ),
        # Only keep `decoder` tree, which means we throw away optimizer.
        scope={"decoder": "decoder"},
        mesh_axis_names=mesh_axis_names,
        mesh_shape=mesh_shape,
    )
    init_state_builder = state_builder.RestoreAndConvertBuilder.default_config().set(
        builder=state_builder.TensorStoreStateStorageBuilder.default_config().set(
            validation=CheckpointValidationType.CONTAINS_STATE_UP_TO_DTYPE,
            dir=checkpoint_dir,
        ),
        converter=converter,
    )
    return init_state_builder


def get_inference_runner(name: str, param_dtype: jnp.dtype) -> InferenceRunner:
    """Make an inference runner initialized with pre-trained state according to model name."""
    # Get the trainer configuration by name.
    ckpt_dir = MODEL_INFO[name]["checkpoint_dir"]

    # If we don't have a local version, first download it.
    local_ckpt_dir = Path(LOCAL_MODEL_DIR) / ckpt_dir
    if not local_ckpt_dir.exists():
        remote_ckpt_dir = os.path.join(REMOTE_MODEL_DIR, ckpt_dir)
        print(f"Copying checkpoint from {remote_ckpt_dir} to {local_ckpt_dir}.")
        os.makedirs(local_ckpt_dir, exist_ok=True)
        os.makedirs(local_ckpt_dir / "gda", exist_ok=True)
        # We only copy the weights, not `gda/learner`, which contains the full optimizer state.
        subprocess.run(["gsutil", "-m", "cp", "-r", os.path.join(remote_ckpt_dir, "tf_*"), local_ckpt_dir])
        subprocess.run(["gsutil", "-m", "cp", "-r", os.path.join(remote_ckpt_dir, "gda", "model"), local_ckpt_dir / "gda"])
        subprocess.run(["gsutil", "-m", "cp", "-r", os.path.join(remote_ckpt_dir, "gda", "prng_key"), local_ckpt_dir / "gda"])
        subprocess.run(["gsutil", "cp", os.path.join(remote_ckpt_dir, "index"), local_ckpt_dir])

    config_name = MODEL_INFO[name]["config_name"]
    config_module = MODEL_INFO[name]["config_module"]
    mesh_axis_names = (
        "data",
        "expert",
        "fsdp",
        "model",
        "seq",
    )
    mesh_shape = (
        1,
        1,
        1,
        len(jax.devices()),
        1,
    )

    trainer_cfg = get_named_trainer_config(
        config_name=config_name,
        config_module=config_module,
    )()

    # Do not load optimizer state, to speed up loading.
    init_state_builder = _init_state_builder_discard_optimizer(
        source_config_name=config_name,
        source_config_module=config_module,
        mesh_axis_names=mesh_axis_names,
        mesh_shape=mesh_shape,
        checkpoint_dir=str(local_ckpt_dir),
    )

    inference_runner_cfg = InferenceRunner.default_config().set(
        name=f"{name}_inference_runner",
        mesh_axis_names=mesh_axis_names,
        mesh_shape=mesh_shape,
        model=trainer_cfg.model.set(dtype=param_dtype),
        input_batch_partition_spec=DataPartitionType.REPLICATED,  # FULL, REPLICATED
        init_state_builder=init_state_builder,
    )
    print(f"Loading state for {name} from:\n{local_ckpt_dir}")
    inference_runner = inference_runner_cfg.instantiate(parent=None)
    return inference_runner


def get_vocab(name: str) -> seqio.Vocabulary:
    """Get the vocabulary based on the model's name."""
    with set_data_dir(DATA_DIR):
        vocab = common.vocab(
            sentencepiece_model_name=MODEL_INFO[name]["sentencepiece_model_name"]
        )
    return vocab


# Load the model checkpoint.
model_name = "7b-sigmoid"  # "7b-softmax"
inference_runner = get_inference_runner(model_name, param_dtype=jnp.bfloat16)
vocab = get_vocab(model_name)

Loading state for 7b-sigmoid from:
downloads/models/gala-7B-sigmoid-hybridnorm-alibi-sprp-2024-12-03-1002/checkpoints/step_00250000


In [9]:
def _preprocess_text(text: str) -> str:
    """Preprocesses text for tokenization.

    Our sentencepiece tokenizers have been trained to see <n> in place of \n.
    "\n" will still work in most places as intended, but it's not always guaranteed
    to tokenize to an individual token, unlike <n>.
    """
    return text.replace("\n", "<n>")


def _postprocess_text(text: str) -> str:
    """Postprocesses text after LM inference.

    Changes <n> back to \n. This is the opposite operation of preprocess_text.
    """
    return text.replace("<n>", "\n")


def _preprocess_inference(
    contexts_list: List[str],
    vocab: seqio.Vocabulary,
    *,
    max_seq_len: int = 256,
    batch_size: int = 1,
) -> Iterator[utils.NestedTensor]:
    batched_remainder = len(contexts_list) % batch_size
    if batched_remainder:
        contexts_list += [[""]] * (batch_size - batched_remainder)
    for chunk_ix in range(0, len(contexts_list), batch_size):
        chunk = contexts_list[chunk_ix : chunk_ix + batch_size]
        output_buffer = []
        for context in chunk:
            output = vocab.encode(_preprocess_text(context))
            output += [vocab.pad_id] * (max_seq_len - len(output))
            output_buffer.append(output)
        yield dict(input_ids=jnp.asarray(output_buffer, dtype=jnp.int32))


def postprocess_inference(
    output: utils.NestedTensor,
    vocab: seqio.Vocabulary,
) -> List[Tuple[str, str]]:
    input_ids = utils.replicate_to_local_data(output["inputs"]["input_ids"])
    # [batch_size, seq_len, vocab_size]
    predicted_logits = utils.replicate_to_local_data(output["outputs"]["logits"])
    results = []
    for ix, in_val in enumerate(input_ids):
        if jnp.all(in_val == vocab.pad_id):
            # Skip fully padded input strings.
            continue
        if in_val[0] == vocab.eos_id:
            # Skip first EOS if it exists.
            in_val_eos_lstrip = in_val[1:]
        else:
            in_val_eos_lstrip = in_val
        input_str = _postprocess_text(vocab.decode(in_val_eos_lstrip))
        input_str_per_token = [_postprocess_text(vocab.decode([input_id])) for input_id in in_val]
        batch_predicted_logits = predicted_logits[ix]
        decoded_continuations = []
        speculative_continuation = ""
        speculative_continuation_per_token = []
        token_prediction_index = len(jnp.argwhere(in_val != vocab.pad_id)) - 1
        new_input_tokens = list(in_val[: token_prediction_index + 1])
        tokens = jnp.argmax(batch_predicted_logits, axis=-1)
        continuation_tokens = tokens
        decoded_continuations.append(_postprocess_text(vocab.decode(tokens)))
        decoded_continuations_per_token = [
            _postprocess_text(vocab.decode([token])) for token in tokens
        ]
        # Add next token info (for self speculative decoding setup).
        head_token_prediction = tokens[token_prediction_index]
        next_token_str = _postprocess_text(vocab.decode([head_token_prediction]))
        speculative_continuation += next_token_str
        speculative_continuation_per_token.append(next_token_str)
        new_input_tokens += [head_token_prediction]
        results.append(
            {
                "input_str": input_str,
                "input_str_per_token": input_str_per_token,
                "decoded_continuations": decoded_continuations,
                "decoded_continuations_per_token": decoded_continuations_per_token,
                "input": output["inputs"]["input_ids"],
                "output": continuation_tokens,
                "speculative_continuation": speculative_continuation,
                "speculative_continuation_per_token": speculative_continuation_per_token,
                "new_input_tokens": new_input_tokens,
                "new_input_str": _postprocess_text(vocab.decode(new_input_tokens)),
                "next_logits": batch_predicted_logits[token_prediction_index],
            }
        )
    return results


def decode_lm_response(
    vocab: seqio.Vocabulary,
    inference_runner: InferenceRunner,
    contexts_list: List[str],
    *,
    max_seq_len: int = 256,
    batch_size: int = 1,
    print_output: bool = False,
):
    results = []
    for batch in inference_runner.run(
        _preprocess_inference(
            contexts_list=contexts_list,
            vocab=vocab,
            max_seq_len=max_seq_len,
            batch_size=batch_size,
        ),
        method="predict",
        prng_key=jax.random.PRNGKey(11),
    ):
        for result in postprocess_inference(batch, vocab):
            results.append(result)

    if not print_output:
        return results
    # Print separately, so we always print it at the end of output in notebook.
    for result in results:
        from tabulate import tabulate

        per_token_table = [["Type", *range(len(result["input"][0]))]]
        per_token_table.append(["input"] + list(result["input"][0]))
        per_token_table.append(["input_str"] + result["input_str_per_token"])
        per_token_table.append([f"prediction"] + list(result["output"]))
        per_token_table.append([f"prediction_str"] + result["decoded_continuations_per_token"])

        # Also show empty chars.
        per_token_table_print = []
        for token_values in per_token_table:
            token_values_print = []
            for token in token_values:
                if isinstance(token, str):
                    token = token.replace("\n", "<n>")
                    if token.isspace():
                        token = f"<{token}>"
                token_values_print.append(token)
            per_token_table_print.append(token_values_print)
        print(tabulate(per_token_table_print, headers="firstrow", tablefmt="grid"))

        print()
        for k, v in result.items():
            print(f"{k}: {v}")
    return results

In [10]:
# Single token-generation:
#  printing all details about intermediate tokens, logits and
#  final next token predicted.
prompt = "Life is like riding a bicycle. To keep"
max_seq_len = 8
_ = decode_lm_response(
    vocab=vocab,
    inference_runner=inference_runner,
    contexts_list=[prompt],
    max_seq_len=max_seq_len,
    print_output=True,
)

+----------------+----------+-----+------+--------+---------+---------+-------+------+------+
| Type           | 0        | 1   | 2    | 3      | 4       | 5       | 6     | 7    | 8    |
+================+==========+=====+======+========+=========+=========+=======+======+======+
| input          | 4585     | 319 | 593  | 8152   | 262     | 14658   | 31905 | 1446 | 1137 |
+----------------+----------+-----+------+--------+---------+---------+-------+------+------+
| input_str      | Life     | is  | like | riding | a       | bicycle | .     | To   | keep |
+----------------+----------+-----+------+--------+---------+---------+-------+------+------+
| prediction     | 10019    | 262 | 262  | 262    | 14658   | 31905   | 749   | 2254 | 380  |
+----------------+----------+-----+------+--------+---------+---------+-------+------+------+
| prediction_str | Sciences | a   | a    | a      | bicycle | .       | You   | fall | your |
+----------------+----------+-----+------+--------+---------

In [12]:
# Generate multiple tokens sequentially.
# This is quite slow+naive, but helps to understand the autoregressive setup.
original_prompt = "Life is like riding a bicycle. To keep"
max_seq_len = 20

prompt = original_prompt
pbar = tqdm(total=max_seq_len)
while (n_tokens := len(vocab.encode(_preprocess_text(prompt)))) < max_seq_len:
    pbar.update(n_tokens - pbar.n)
    pbar.set_description(prompt[len(original_prompt):].replace("\n", "\\n"))
    lm_response = decode_lm_response(
        vocab=vocab,
        inference_runner=inference_runner,
        contexts_list=[prompt],
        max_seq_len=max_seq_len,
        print_output=False,
    )
    prompt = lm_response[0]["new_input_str"]
pbar.close()
print()
print(prompt)
    

 your balance, you must keep moving.\n—:  95%|█████████▌| 19/20 [01:03<00:03,  3.32s/it]

Life is like riding a bicycle. To keep your balance, you must keep moving.
— Albert
